# BAA10Y analyst agent - minimal smoke notebook

This notebook gives the smallest runnable path to exercise the **analyst agent we built**. It is intentionally minimal: one config cell, one interactive text run, and one structured forecast run.

The goal here is not benchmarking. It is just to confirm the analyst agent loads, receives the expected payload, and can produce a response.

In [ ]:
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parents[1]
load_dotenv(ROOT / ".env", override=False)

AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"

RUN_AGENT = True
HORIZON = 5

from aieng.forecasting.evaluation.task import ForecastingTask
from BAA10Y_forecasting import build_baa10y_multivariate_service
from BAA10Y_forecasting.data import (
    DEFAULT_COVARIATE_SERIES_IDS,
    HYOAS_OPTIONAL_COVARIATE_SERIES_IDS,
    baa10y_change_series_id,
)
from BAA10Y_forecasting.analyst_agent.agent import (
    BAA10YForecastPromptBuilder,
    build_baa10y_agent_predictor,
    build_baa10y_code_exec_config,
    build_baa10y_tool_config,
)

print("RUN_AGENT =", RUN_AGENT, "| model =", AGENT_MODEL, "| horizon =", HORIZON)

from aieng.forecasting.methods.numerical import DartsLinearRegressionPredictor

import matplotlib.pyplot as plt


## 1. Build the configs

Use the code-exec config for interactive exploration and the forecast-tool config for the most reliable structured forecast smoke test.

In [ ]:
chat_config = build_baa10y_code_exec_config(model=AGENT_MODEL)

forecast_config = build_baa10y_tool_config(model=AGENT_MODEL)

print("Chat agent:          ", chat_config.name)
print("  search enabled:    ", chat_config.context_retrieval.enabled)
print("  code-exec enabled: ", chat_config.code_execution.enabled)
print("  skills loaded:     ", [p.name for p in chat_config.skills_dirs])

print("Forecast agent:      ", forecast_config.name)
print("  search enabled:    ", forecast_config.context_retrieval.enabled)
print("  function tools:    ", [getattr(t, "name", type(t).__name__) for t in forecast_config.function_tools])

print("--- Chat instruction preview ---\n")
print(chat_config.instruction[:1200], "...")


## 2. Inspect the prompt payload

Before running the model, build one real payload and inspect its keys. This confirms the notebook is exercising the current analyst-agent contract rather than the older starter-agent contract.

In [ ]:
COVARIATES = [
    *DEFAULT_COVARIATE_SERIES_IDS,
    *HYOAS_OPTIONAL_COVARIATE_SERIES_IDS,
]

svc = build_baa10y_multivariate_service(covariate_series_ids=COVARIATES)
target_series_id = baa10y_change_series_id(HORIZON)
full = svc.get_series(target_series_id, as_of=datetime.now(tz=timezone.utc).replace(tzinfo=None))
full["timestamp"] = pd.to_datetime(full["timestamp"])
last_date = full["timestamp"].iloc[-1]
AS_OF = last_date - pd.offsets.BDay(HORIZON + 1)

task = ForecastingTask(
    task_id=f"baa10y_analyst_smoke_{HORIZON}b",
    target_series_id=target_series_id,
    horizons=[HORIZON],
    frequency="B",
    description=f"BAA10Y cumulative spread change in basis points, {HORIZON} business days ahead.",
)
ctx = svc.context(as_of=AS_OF)
payload = BAA10YForecastPromptBuilder()(task=task, context=ctx)

import json
payload_obj = json.loads(payload)
print(sorted(payload_obj.keys()))
print()
print("target_series_id:", payload_obj["target_series_id"])
print("target_window_business_days:", payload_obj["target_window_business_days"])
print("horizons:", payload_obj["horizons"])
print("missing_covariates count:", len(payload_obj["missing_covariates"]))
print("target_history_csv head:")
print("\n".join(payload_obj["target_history_csv"].splitlines()[:4]))
print()
print("daily_change_history_csv head:")
print("\n".join(payload_obj["daily_change_history_csv"].splitlines()[:4]))


## 3. Open-ended smoke run

This checks that the agent can load as a conversational analyst. Leave `RUN_AGENT = False` until your environment is ready.

In [ ]:
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import AdkTextRunner, AdkTextRunnerConfig

QUESTION = (
    "Summarize the main forces that would matter for a near-term BAA10Y spread-change forecast. "
    "Be concise and distinguish direction from uncertainty."
)

if RUN_AGENT:
    chat_agent = build_adk_agent(chat_config)
    runner = AdkTextRunner(chat_agent, config=AdkTextRunnerConfig(app_name="baa10y_analyst_smoke_chat"))
    reply = await runner.run_text_async(QUESTION)  # noqa: F704, PLE1142
    print(reply)
else:
    print("RUN_AGENT is False - set it to True to run the conversational smoke test.")


## 4. Retrospective structured smoke run

This runs the analyst predictor on one already-resolved origin using the forecast-tool config. It is a backward-looking smoke test of the structured forecasting path.

In [ ]:
if RUN_AGENT:
    predictor = build_baa10y_agent_predictor(forecast_config)
    pred = predictor.predict(task, ctx)[0]
    fc = pred.payload

    rows = full[full["timestamp"] >= AS_OF + pd.offsets.BDay(HORIZON)]
    actual = float(rows["value"].iloc[0]) if not rows.empty else None
    lo, hi = fc.quantiles[0.10], fc.quantiles[0.90]

    print(f"Origin as_of={AS_OF.date()}  horizon={HORIZON}b  (latest data {last_date.date()})\n")
    print(f"point forecast: {fc.point_forecast:+.2f} bps")
    print(f"80% interval : [{lo:+.2f}, {hi:+.2f}] bps")
    print(f"median check : {fc.quantiles[0.50]:+.2f} bps")
    if actual is None:
        print("actual       : N/A")
    else:
        in_band = "yes" if lo <= actual <= hi else "no"
        print(f"actual       : {actual:+.2f} bps  | in 80% band? {in_band}")
    if pred.metadata.get("rationale"):
        print("\nRationale preview:")
        print(pred.metadata["rationale"][:600])
else:
    print("RUN_AGENT is False - set it to True to run the structured forecast smoke test.")


## 5. Live prediction run

This uses the latest available observation as the forecast origin and produces a current prediction with no realized-value comparison.

In [ ]:
if RUN_AGENT:
    live_as_of = last_date
    live_ctx = svc.context(as_of=live_as_of)
    live_task = ForecastingTask(
        task_id=f"baa10y_analyst_live_{HORIZON}b",
        target_series_id=target_series_id,
        horizons=[HORIZON],
        frequency="B",
        description=f"BAA10Y cumulative spread change in basis points, {HORIZON} business days ahead.",
    )
    predictor = build_baa10y_agent_predictor(forecast_config)
    pred = predictor.predict(live_task, live_ctx)[0]
    fc = pred.payload
    lo, hi = fc.quantiles[0.10], fc.quantiles[0.90]

    print(f"Live forecast origin as_of={live_as_of.date()}  horizon={HORIZON}b\n")
    print(f"point forecast: {fc.point_forecast:+.2f} bps")
    print(f"80% interval : [{lo:+.2f}, {hi:+.2f}] bps")
    print(f"median check : {fc.quantiles[0.50]:+.2f} bps")
    if pred.metadata.get("rationale"):
        print("\nRationale preview:")
        print(pred.metadata["rationale"][:600])
else:
    print("RUN_AGENT is False - set it to True to run the live prediction.")


## 6. Live comparison: agent vs linear regression (no covariates)

This compares the live analyst-agent forecast to a univariate linear-regression baseline using the same live origin and horizon. The baseline uses target lags only (`covariate_series_ids=None`).

In [ ]:
if RUN_AGENT:
    # Agent forecast on live origin
    live_as_of = last_date
    live_ctx = svc.context(as_of=live_as_of)
    live_task = ForecastingTask(
        task_id=f"baa10y_compare_live_{HORIZON}b",
        target_series_id=target_series_id,
        horizons=[HORIZON],
        frequency="B",
        description=f"BAA10Y cumulative spread change in basis points, {HORIZON} business days ahead.",
    )

    agent_pred = build_baa10y_agent_predictor(forecast_config).predict(live_task, live_ctx)[0]
    agent_fc = agent_pred.payload
    agent_q10 = agent_fc.quantiles[0.10]
    agent_q50 = agent_fc.quantiles[0.50]
    agent_q90 = agent_fc.quantiles[0.90]

    # Linear regression baseline (target-only, no covariates)
    linreg = DartsLinearRegressionPredictor(
        lags=max(21, HORIZON),
        covariate_series_ids=None,
        num_samples=200,
    )
    linreg_pred = linreg.predict(live_task, live_ctx)[0]
    linreg_fc = linreg_pred.payload
    linreg_q10 = linreg_fc.quantiles[0.10]
    linreg_q50 = linreg_fc.quantiles[0.50]
    linreg_q90 = linreg_fc.quantiles[0.90]

    print(f"Live origin: {live_as_of.date()}  horizon={HORIZON}b")
    print()
    print("Model                         q10      q50      q90    point")
    print(f"Analyst agent            {agent_q10:+7.2f}  {agent_q50:+7.2f}  {agent_q90:+7.2f}  {agent_fc.point_forecast:+7.2f}")
    print(f"Linear regression (no X) {linreg_q10:+7.2f}  {linreg_q50:+7.2f}  {linreg_q90:+7.2f}  {linreg_fc.point_forecast:+7.2f}")

    print()
    print(f"Point spread (agent - linreg): {agent_fc.point_forecast - linreg_fc.point_forecast:+.2f} bps")
else:
    print("RUN_AGENT is False - set it to True to run the live model comparison.")


## 7. Rolling chart: agent vs linear regression vs actual

This section compares both models across multiple resolved forecast origins and plots point forecasts plus 10th/90th bands.

In [ ]:
if RUN_AGENT:
    N_ORIGINS = 30
    predictor_agent = build_baa10y_agent_predictor(forecast_config)
    predictor_linreg = DartsLinearRegressionPredictor(
        lags=max(21, HORIZON),
        covariate_series_ids=None,
        num_samples=200,
    )

    candidate_origins = full["timestamp"].iloc[:-HORIZON]
    origin_dates = candidate_origins.tail(N_ORIGINS).tolist()
    rows = []

    for origin in origin_dates:
        bt_task = ForecastingTask(
            task_id=f"baa10y_compare_{HORIZON}b_{origin.date()}",
            target_series_id=target_series_id,
            horizons=[HORIZON],
            frequency="B",
            description=f"BAA10Y cumulative spread change in basis points, {HORIZON} business days ahead.",
        )
        bt_ctx = svc.context(as_of=origin)

        agent_out = predictor_agent.predict(bt_task, bt_ctx)[0].payload
        linreg_out = predictor_linreg.predict(bt_task, bt_ctx)[0].payload

        actual_rows = full[full["timestamp"] >= origin + pd.offsets.BDay(HORIZON)]
        if actual_rows.empty:
            continue
        actual_value = float(actual_rows["value"].iloc[0])

        rows.append({
            "origin": pd.Timestamp(origin),
            "actual": actual_value,
            "agent_point": float(agent_out.point_forecast),
            "agent_q10": float(agent_out.quantiles[0.10]),
            "agent_q90": float(agent_out.quantiles[0.90]),
            "linreg_point": float(linreg_out.point_forecast),
            "linreg_q10": float(linreg_out.quantiles[0.10]),
            "linreg_q90": float(linreg_out.quantiles[0.90]),
        })

    comp_df = pd.DataFrame(rows).sort_values("origin")
    if comp_df.empty:
        print("No resolved origins available for plotting.")
    else:
        fig, ax = plt.subplots(figsize=(12, 6))

        ax.plot(comp_df["origin"], comp_df["actual"], color="black", linewidth=2.0, label="Actual")

        ax.plot(comp_df["origin"], comp_df["agent_point"], color="#1f77b4", linewidth=1.8, label="Agent point")
        ax.fill_between(
            comp_df["origin"],
            comp_df["agent_q10"],
            comp_df["agent_q90"],
            color="#1f77b4",
            alpha=0.18,
            label="Agent q10-q90",
        )

        ax.plot(comp_df["origin"], comp_df["linreg_point"], color="#ff7f0e", linewidth=1.8, label="Linear regression point")
        ax.fill_between(
            comp_df["origin"],
            comp_df["linreg_q10"],
            comp_df["linreg_q90"],
            color="#ff7f0e",
            alpha=0.18,
            label="Linear regression q10-q90",
        )

        ax.set_title(f"BAA10Y {HORIZON}b forecasts: agent vs linear regression (last {len(comp_df)} resolved origins)")
        ax.set_xlabel("Forecast origin date")
        ax.set_ylabel("BAA10Y cumulative change (bps)")
        ax.axhline(0.0, color="gray", linewidth=1.0, alpha=0.7)
        ax.legend(loc="best")
        ax.grid(alpha=0.25)
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

        print(f"Plotted {len(comp_df)} resolved origins for horizon={HORIZON}b.")
else:
    print("RUN_AGENT is False - set it to True to run and plot the rolling comparison.")


## 8. Next step

Once this notebook runs, the next useful step is to add a tiny multi-origin smoke loop or a dedicated test file. For now, this notebook gives both a retrospective smoke test and a live prediction path for the analyst agent.